## Story Idea Generation Demo

#### This demo is to run text generation for the best GRU, LSTM, and custom architecture outside of the model training space.

In [19]:
import os
import torch
import string
from ModelArchitecture.CharRNN import CharRNNClassic, CharRNNCustom

In [20]:
allowed_chars = [' ', '!', '"', '#', '$', '%', '&', "'", '(', ')', '*', '+', ',', '-', '.', '/', '0', '1', '2', '3', '4', '5', '6', '7', '8', '9', ':', ';', '<', '=', '>', '?', '@', 'A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y', 'Z', '[', '\\', ']', '_', '`', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z', '{', '|', '}', '~']

chars = sorted(allowed_chars)
vocab_size = len(chars)
print(vocab_size)

char_to_idx = {c:i for i,c in enumerate(chars)}
idx_to_char = {i:c for i,c in enumerate(chars)}

94


In [21]:
def generate_text_custom_model(model, char_to_idx, idx_to_char, max_length=2000, start_text = "<", stop_char = ">", temperature=0.8, device="cpu"):
    
    model.eval()
    
    input_seq = torch.tensor([[char_to_idx[c] for c in start_text]], device=device)
    hidden = model.init_hidden(1, device=device)
    
    
    input_seq = torch.tensor([[char_to_idx[c] for c in start_text]], device=device)  # (1, seq_len)
    hidden = model.init_hidden(1, device=device)

    # Prime the model
    for i in range(len(start_text) - 1):
        _, hidden = model.forward2(input_seq[:, i:i+1], hidden)  # slice instead of index to keep 2D

    current_char = input_seq[:, -1:]  # (1, 1) instead of (1,)

    generated = start_text
    for _ in range(max_length):
        output, hidden = model.forward2(current_char, hidden)

        logits = output.squeeze() / temperature
        probs = torch.softmax(logits, dim=0)

        next_idx = torch.multinomial(probs, 1).item()
        next_char = idx_to_char[next_idx]

        generated += next_char
        if next_char == stop_char:
            break
        current_char = torch.tensor([[next_idx]], device=device)  # (1, 1) instead of (1,)
    """
    # Prime the model with the seed text
    for i in range(len(start_text) - 1):
        _, hidden = model.forward2(input_seq[:, i], hidden)

    current_char = input_seq[:, -1]

    generated = start_text
    for _ in range(max_length):

        output, hidden = model.forward2(current_char, hidden)

        logits = output.squeeze() / temperature
        probs = torch.softmax(logits, dim=0)

        next_idx = torch.multinomial(probs, 1).item()
        next_char = idx_to_char[next_idx]

        generated += next_char
        if(next_char == stop_char):
            break;
        current_char = torch.tensor([next_idx], device=device)

    """
    return generated


def generate_text_classic(model, char_to_idx, idx_to_char, max_length=2000, start_text = "<", stop_char = ">", temperature=0.8, device="cpu"):
    
    model.eval()
    
    input_seq = torch.tensor([[char_to_idx[c] for c in start_text]], device=device)
    hidden = model.init_hidden(1)
    
    if isinstance(hidden, tuple):
        hidden = (hidden[0].to(device), hidden[1].to(device))
    else:
        hidden = hidden.to(device)

    generated = start_text

    # Prime the model with the seed text
    for i in range(len(start_text) - 1):
        _, hidden = model.forward2(input_seq[:, i], hidden)

    current_char = input_seq[:, -1]

    for _ in range(max_length):

        output, hidden = model.forward2(current_char, hidden)

        logits = output.squeeze() / temperature
        probs = torch.softmax(logits, dim=0)

        next_idx = torch.multinomial(probs, 1).item()
        next_char = idx_to_char[next_idx]

        generated += next_char
        if(next_char == stop_char):
            break;
        current_char = torch.tensor([next_idx], device=device)


    return generated

In [22]:
if torch.cuda.is_available():
    device = torch.device("cuda:0")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

print(device)

cuda:0


GRU MODEL DEMO

In [23]:
#Change these parameters if you'd like text generation to be slightly different
temperature = 0.8
max_char_size = 2000
start_text = "<"

In [24]:
model_path = os.path.join("best_model_weights", "classic_rnn_gru_epoch_11_2026-03-17_19-18-39.pt")
checkpoint = torch.load(model_path, weights_only=True)
gru_model = CharRNNClassic(vocab_size, 512, vocab_size, embedding_size=128, model="gru", n_layers=2, batch_first=True)
gru_model.load_state_dict(checkpoint)
gru_model.to(device)
generate_text_classic(gru_model, char_to_idx=char_to_idx, idx_to_char=idx_to_char, max_length=max_char_size, start_text=start_text, temperature=temperature, device=device)

"<At this point that the Army agrees to take Ravenpaw, who are married the news that he gives everyone ever since then Anna. Elizabeth Larry is stabbed in the library. As a result, he does not separate. After he and Simon also learn a dinner in the flume and kills Marina, training during the novel. The book includes a squad of a mountain range of Bloom Layton and an empty crisis. The corridors have accidentally forgettends that the children retreat to his mother, without ever watching her children to visit her daughters. The second part is made to see a man named Belicha, the doctor and a werewolf rare sensation to convince him to steal control of their experiments. Alma, however, drugs her considerable to the situation. The men have blowing them a fear from Jane Aurora, who has her past and his assistant Night and later becomes infected and the assistant presents a barrier named Goth. Respan is helped by the legendary Helkon Legion she never found him and save her. She's now working f

LSTM MODEL DEMO

In [25]:
#Change these parameters if you'd like text generation to be slightly different
temperature = 0.8
max_char_size = 2000
start_text = "<"

In [26]:
model_path = os.path.join("best_model_weights", "classic_rnn_lstm_epoch_15_2026-03-17_19-48-54.pt")
checkpoint = torch.load(model_path, weights_only=True)
lstm_model = CharRNNClassic(vocab_size, 512, vocab_size, embedding_size=128, model="lstm", n_layers=2, batch_first=True)
lstm_model.load_state_dict(checkpoint)
lstm_model.to(device)
generate_text_classic(lstm_model, char_to_idx=char_to_idx, idx_to_char=idx_to_char, max_length=max_char_size, start_text=start_text, temperature=temperature, device=device)

"<Danny Alexander and Paul Grace of Anzie are attempted with with the freedom of making strange string postels in the light of Andelaire as they are dragged out of life with being drawn into space transmitting time (Arac runner before their friend had blamed him. They prepare to leave the role in their sisters. Even when their brother is horribly more, and Proctor rushes home afterwards, Avey is angry when she ends up resistant to the man who is shocked with her custody and becomes an interest in her children, and is told from the main character. In several sleuches when she meets her but was once again comforted, for who he had left her. Victorial confesses she is a young couple of canyon and injures her to lower fair wanting her like an Army. Afterwards, including her father, he quickly becomes grateful for her, and the inhabitants accept leading to soldier who restricted her. He finds out that they will sit up with Mrs Mandy's concern from the new generation. He meets and kills her,

CUSTOM ARCHITECTURE MODEL DEMO

In [32]:
#Change these parameters if you'd like text generation to be slightly different
temperature = 0.8
max_char_size = 2000
start_text = "<"

In [33]:
model_path = os.path.join("best_model_weights", "custom_rnn_trial_24_epoch_22_2026-03-19_12-58-39.pt")
checkpoint = torch.load(model_path, weights_only=True)
gru_model = CharRNNCustom(vocab_size, vocab_size, gru_hidden_sizes=[128,256,512], lstm_hidden_size=1024, embedding_size=128, batch_first=True)
gru_model.load_state_dict(checkpoint)
gru_model.to(device)
generate_text_custom_model(gru_model, char_to_idx=char_to_idx, idx_to_char=idx_to_char, max_length=max_char_size, start_text=start_text, temperature=temperature, device=device)

"<A young black haired young woman is a birthday party because she wants to ensure the mother, and her stepmother has just told her some guests about her husband, and Elizabeth Roo who is soon her aunt and as Pennyroyal. They transport a turn for the high priestess, and tells her they are still trying to use the house to keep her unharmed. She returns to the city of the Golden Rockets, where she meets her ghost. A witch comes to her mother's new boat, Annabelle and Felicia Judge When they find it is Harriet's injured kiss. This may be able to confront the suspicion, and she quickly decides to send her to her death. Not previously doing this this lives in the past. She has a secret, specialized with no flight and a series of little money from school. Miss Binney has been chosen as being a decomposed politics, which helped her ignore her in single control. Meanwhile, Stevens starts working at that adventure for the help of Tanaka (Tatiana Maplestam) and Billy Lowel (no leader of the sewe